<a href="https://colab.research.google.com/github/icsl-aist/hsr-genesis/blob/main/examples/tutorials/7_troubleshoot_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Troubleshooting Guide / トラブルシューティング

> **Objective / 目的**
>
> This notebook helps you diagnose and fix common issues when running the HSR Genesis tutorials on Google Colab.
>
> このノートブックは、Google Colab で HSR Genesis チュートリアルを実行する際の一般的な問題の診断と解決に役立ちます。


## Quick Diagnostics / クイック診断

Run the cell below to check your environment. It will report any issues found.

下のセルを実行して環境をチェックしてください。見つかった問題が報告されます。


In [ ]:
import sys, os, subprocess, importlib

issues = []

print('=' * 60)
print('HSR Genesis — Environment Diagnostics')
print('=' * 60)

# 1. Python version
print(f'\n[1] Python: {sys.version.split()[0]}')
if sys.version_info < (3, 10):
    issues.append('Python < 3.10. hsr-genesis requires Python >= 3.10.')

# 2. GPU availability
print('\n[2] GPU check:')
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=5)
    if result.returncode == 0 and result.stdout.strip():
        print(f'    GPU: {result.stdout.strip()}')
    else:
        print('    No GPU detected!')
        issues.append('No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')
except Exception:
    print('    nvidia-smi not found.')
    issues.append('nvidia-smi not found. You may not be on a GPU runtime.')

# 3. Key packages
print('\n[3] Package check:')
for pkg, import_name in [('genesis-world', 'genesis'), ('mediapy', 'mediapy'), ('torch', 'torch')]:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'    {pkg}: {ver} ✓')
    except ImportError:
        print(f'    {pkg}: NOT INSTALLED ✗')
        issues.append(f'{pkg} not installed. Run setup_colab() or: !pip install {pkg}')

# 4. setuptools version
print('\n[4] setuptools:')
try:
    import setuptools
    sv = setuptools.__version__
    print(f'    setuptools {sv}')
    if int(sv.split('.')[0]) >= 82:
        issues.append(f'setuptools {sv} >= 82. Run: !pip install "setuptools<82"')
except Exception:
    print('    setuptools not found')

# 5. hsr_genesis
print('\n[5] hsr_genesis:')
try:
    import hsr_genesis
    print(f'    Loaded from: {hsr_genesis.__file__}')
except ImportError as e:
    print(f'    import failed: {e}')
    issues.append('hsr_genesis not importable. Run setup_colab() or restart runtime.')

# 6. URDF and meshes
print('\n[6] Data files:')
for repo_dir in ['/content/hsr-genesis', os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if '__file__' in dir() else '.']:
    urdf = os.path.join(repo_dir, 'data/urdf/hsrb4s.urdf')
    meshes = os.path.join(repo_dir, 'data/urdf/hsrb_meshes')
    if os.path.exists(urdf):
        print(f'    URDF: {urdf} ✓')
        if os.path.exists(meshes) and os.listdir(meshes):
            print(f'    Meshes: {meshes} ✓')
        else:
            print(f'    Meshes: {meshes} EMPTY/MISSING ✗')
            issues.append('hsrb_meshes submodule not fetched. Run: !git -C /content/hsr-genesis submodule update --init --recursive')
        break
else:
    print('    URDF not found.')
    issues.append('URDF not found. Run setup_colab() to clone the repo.')

# 7. EGL ICD
print('\n[7] EGL ICD config:')
icd_path = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if os.path.exists(icd_path):
    print(f'    {icd_path} exists ✓')
else:
    print(f'    {icd_path} MISSING ✗')
    issues.append('EGL ICD config missing. Run setup_colab() or write the ICD file manually.')

# Summary
print('\n' + '=' * 60)
if issues:
    print(f'FOUND {len(issues)} ISSUE(S):')
    for i, issue in enumerate(issues, 1):
        print(f'  {i}. {issue}')
else:
    print('All checks passed! Your environment looks good.')
print('=' * 60)

---

## Common Issues / 一般的な問題


### 1. No GPU detected / GPU が検出されない

**Symptom:** `nvidia-smi: command not found` or `GenesisException: GPU backend unavailable`.

**Fix:**
1. Go to **Runtime → Change runtime type**
2. Set **Hardware accelerator** to **T4 GPU** (or better)
3. Click **Save** — Colab will restart the runtime
4. Re-run the setup cell

**症状:** `nvidia-smi: command not found` または `GenesisException: GPU backend unavailable`。

**解決策:**
1. **ランタイム → ランタイムのタイプを変更**
2. **ハードウェア アクセラレータ** を **T4 GPU** に設定
3. **保存** をクリック（ランタイムが再起動されます）
4. セットアップセルを再実行


### 2. `ModuleNotFoundError: No module named 'hsr_genesis'`

**Cause:** The repo wasn't installed or the kernel needs a restart after install.

**Fix:**

Option A — Re-run `setup_colab()`:
```python
from hsr_genesis.tutorial_utils import setup_colab
setup_colab()
```

Option B — Manual install + restart:
```python
!pip install -e /content/hsr-genesis
```
Then go to **Runtime → Restart runtime** and re-run the cells.

Option C — PYTHONPATH fallback (no restart needed):
```python
import sys
sys.path.insert(0, '/content/hsr-genesis/src')
import hsr_genesis
```


### 3. `setuptools >= 82` conflict with torch

**Symptom:** `torch 2.x requires setuptools<82, but you have setuptools 82.x`.

**Fix:**
```python
!pip install 'setuptools<82'
```
Then **Runtime → Restart runtime**.

`setup_colab()` handles this automatically, but if you installed packages manually you may need to do this.


### 4. Meshes missing (`hsrb_meshes` submodule not fetched)

**Symptom:** `FileNotFoundError` or URDF fails to load because mesh files are missing.

**Fix:**
```python
!git -C /content/hsr-genesis submodule update --init --recursive --depth 1
```

Or re-clone with submodules:
```python
!rm -rf /content/hsr-genesis
!git clone --depth 1 --recurse-submodules --shallow-submodules https://github.com/icsl-aist/hsr-genesis.git /content/hsr-genesis
```


### 5. `GenesisException: Genesis already initialized`

**Cause:** You re-ran `gs.init()` (or `init_sim()`) without restarting the kernel.

**Fix:** `init_sim()` is idempotent — just skip re-running it. If you need a fresh scene, use:
```python
reset_sim()
```

If Genesis itself needs re-init, go to **Runtime → Restart runtime**.


### 6. EGL / rendering errors (black frames, EGL not initialized)

**Symptom:** `cam.render()` returns black images, or EGL-related errors.

**Fix:**

1. Verify the EGL ICD config exists:
```python
import os
print(os.path.exists('/usr/share/glvnd/egl_vendor.d/10_nvidia.json'))
```

2. If missing, write it manually:
```python
icd = '{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}'
os.makedirs('/usr/share/glvnd/egl_vendor.d', exist_ok=True)
with open('/usr/share/glvnd/egl_vendor.d/10_nvidia.json', 'w') as f:
    f.write(icd)
```

3. Make sure you're on a **GPU runtime** (not CPU).

4. Restart the runtime and re-run.


### 7. `mediapy` video not showing

**Symptom:** `show_video()` runs but nothing displays.

**Fix:**
- Make sure you ran `run()` or `step()` with `render=True` (default) before calling `show_video()`.
- Try `show_frame()` to check if at least one frame was captured.
- If frames list is empty: `print(len(frames))` — if 0, the camera wasn't initialized. Call `init_sim()` first.


### 8. Out of memory (OOM) on GPU

**Symptom:** `CUDA out of memory` error.

**Fix:**
- Reduce camera resolution: `init_sim(cam_res=(320, 240))`
- Reduce simulation substeps: `init_sim(substeps=10)`
- Restart runtime to clear GPU memory: **Runtime → Restart runtime**
- Use a smaller `dt`: `init_sim(dt=0.01)`


---

## Manual Setup (if `setup_colab()` fails) / 手動セットアップ

If the automated `setup_colab()` doesn't work, you can run each step manually:

`setup_colab()` が動作しない場合、各ステップを手動で実行できます:


#### Step 1: Install packages / パッケージのインストール

In [ ]:
!pip install 'setuptools<82' jedi -q
!pip install genesis-world==0.4.6 -q
!pip install mediapy -q

#### Step 2: Clone repo with submodules / サブモジュール付きでリポジトリをクローン

In [ ]:
!rm -rf /content/hsr-genesis
!git clone --depth 1 --recurse-submodules --shallow-submodules https://github.com/icsl-aist/hsr-genesis.git /content/hsr-genesis
!pip install -e /content/hsr-genesis -q

#### Step 3: Add to sys.path and verify / sys.path に追加して確認

In [ ]:
import sys
sys.path.insert(0, '/content/hsr-genesis/src')

import hsr_genesis
print('hsr_genesis loaded from:', hsr_genesis.__file__)

import os
urdf = '/content/hsr-genesis/data/urdf/hsrb4s.urdf'
meshes = '/content/hsr-genesis/data/urdf/hsrb_meshes'
print('URDF exists:', os.path.exists(urdf))
print('Meshes exist:', os.path.exists(meshes) and len(os.listdir(meshes)) > 0)

#### Step 4: Configure EGL / EGL の設定

In [ ]:
import os
icd_path = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
os.makedirs(os.path.dirname(icd_path), exist_ok=True)
with open(icd_path, 'w') as f:
    f.write('{\n    "file_format_version" : "1.0.0",\n    "ICD" : {\n        "library_path" : "libEGL_nvidia.so.0"\n    }\n}\n')
print('EGL ICD config written to', icd_path)

#### Step 5: Restart runtime / ランタイムを再起動

> **Important:** After manual setup, go to **Runtime → Restart runtime** before running the tutorial notebooks.

> **重要:** 手動セットアップ後、チュートリアルノートブックを実行する前に **ランタイム → ランタイムを再起動** してください。


---

## Useful Commands / 便利なコマンド


### Check GPU memory / GPU メモリの確認

In [ ]:
!nvidia-smi

### Check installed package versions / インストール済みパッケージのバージョン確認

In [ ]:
import genesis as gs, torch, mediapy, numpy as np
print(f'genesis:    {gs.__version__}')
print(f'torch:      {torch.__version__}')
print(f'numpy:      {np.__version__}')
print(f'mediapy:    {mediapy.__version__}')
print(f'CUDA avail: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

### Force re-clone the repo / リポジトリを強制再クローン

In [ ]:
# Uncomment to force a fresh clone:
# !rm -rf /content/hsr-genesis
# !git clone --depth 1 --recurse-submodules --shallow-submodules https://github.com/icsl-aist/hsr-genesis.git /content/hsr-genesis
# !pip install -e /content/hsr-genesis -q
print('Uncomment the lines above to force re-clone.')

---

## Still stuck? / それでも解決しない場合？

1. **Restart the runtime:** Runtime → Restart runtime, then re-run all cells from the top.
2. **Factory reset:** Runtime → Factory reset runtime, then re-run setup.
3. **Check the repo:** Visit [github.com/icsl-aist/hsr-genesis](https://github.com/icsl-aist/hsr-genesis) for issues and docs.

1. **ランタイムを再起動:** ランタイム → ランタイムを再起動、最初からすべてのセルを再実行。
2. **ファクトリリセット:** ランタイム → ランタイムをファクトリリセット、セットアップを再実行。
3. **リポジトリを確認:** [github.com/icsl-aist/hsr-genesis](https://github.com/icsl-aist/hsr-genesis) でイシューやドキュメントを確認。
